Attention divides its scores by $\sqrt{d_k}$, where $d_k$ is the dimension of the query and key vectors of one head. In multi-head attention $d_k = d/H$, not the model width $d$.

The reason is the variance of a dot product. Assume the components of a query $q$ and a key $k$ are independent, with mean 0 and variance 1, as they roughly are at initialization. Each product $q_\ell k_\ell$ then has mean 0 and variance 1, and the score is a sum of $d_k$ such terms:

$$
s = q \cdot k = \sum_{\ell=1}^{d_k} q_\ell k_\ell, \qquad \mathbb{E}[s] = 0, \qquad \mathrm{Var}(s) = d_k .
$$

The scores spread out in proportion to $\sqrt{d_k}$. Dividing by $\sqrt{d_k}$ brings the variance back to 1 for any head size:

$$
\mathrm{Var}\!\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = 1 .
$$

The cell checks both variances by sampling 10,000 query and key pairs for each head size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
dims = [8, 16, 32, 64, 128, 256, 512]
var_raw, var_scaled = [], []
for d_k in dims:
    q = rng.standard_normal((10_000, d_k))
    k = rng.standard_normal((10_000, d_k))
    s = (q * k).sum(axis=1)
    var_raw.append(s.var())
    var_scaled.append((s / np.sqrt(d_k)).var())
    print(f"d_k = {d_k:3d}   Var(q.k) = {var_raw[-1]:7.1f}   Var(q.k / sqrt(d_k)) = {var_scaled[-1]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(dims, var_raw, marker="o", label=r"$q \cdot k$")
ax.plot(dims, var_scaled, marker="o", label=r"$q \cdot k / \sqrt{d_k}$")
ax.set_xlabel("$d_k$")
ax.set_ylabel("variance of the score")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The spread matters because the softmax exponentiates the scores. For $d_k = 64$, a common head size, unscaled scores have a standard deviation of 8, so the largest score of a row is far above the others and its weight is close to 1. The weights of all other keys are then close to 0, and so is the gradient of the softmax with respect to every score, which means the model barely learns which keys to attend to. The cell draws one query and 16 keys with $d_k = 64$ and compares the weights with and without the scaling.

In [ ]:
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

d_k, T = 64, 16
q = rng.standard_normal(d_k)
K = rng.standard_normal((T, d_k))
scores = K @ q
w_raw = softmax(scores)
w_scaled = softmax(scores / np.sqrt(d_k))
for name, w in [("unscaled", w_raw), ("scaled", w_scaled)]:
    print(f"{name:9s} largest weight {w.max():.3f}   keys with weight above 0.01: {(w > 0.01).sum()} of {T}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, w, title in zip(axes, [w_raw, w_scaled], [r"$\mathrm{softmax}(q \cdot k)$", r"$\mathrm{softmax}(q \cdot k / \sqrt{d_k})$"]):
    ax.bar(range(T), w)
    ax.set_title(title)
    ax.set_xlabel("key index")
axes[0].set_ylabel("attention weight")
plt.tight_layout()
plt.show()

## PyTorch reference

| PyTorch class | Description |
|---|---|
| [`nn.Linear`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html) | Applies an affine linear transformation to the incoming data: $y = xA^T + b$. |
| [`nn.Softmax`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Softmax.html) | Applies the Softmax function to an n-dimensional input Tensor. |
| [`nn.MultiheadAttention`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.MultiheadAttention.html) | Allows the model to jointly attend to information from different representation subspaces. |